# Raccolta Dati da Reddit — r/Italia, keyword: *estate*
Raccolta tramite **Arctic Shift** (archivio pubblico Reddit per ricerca accademica).
Nessuna credenziale richiesta.

**Strategia**: cerchiamo direttamente i **commenti** che contengono la parola 'estate'
in r/Italia, insieme ai metadati del post padre. Questo è più efficiente che cercare post
e poi scaricare i loro commenti.

## 0. Installazione dipendenze

In [2]:
# !pip install requests spacy tqdm
# !python -m spacy download it_core_news_sm

## 1. Configurazione

In [3]:
import requests
import time
import pandas as pd
from datetime import datetime

SUBREDDIT       = "Italia"
KEYWORD         = "estate"
TARGET_COMMENTS = 700
OUTPUT_CSV      = f"corpus_{SUBREDDIT}_{KEYWORD}.csv"

BASE_URL = "https://arctic-shift.photon-reddit.com/api"
HEADERS  = {"User-Agent": "python:elnsm.progetto.estate:v1.0 (academic NLP project)"}

print("Configurazione pronta.")
print(f"  Subreddit : r/{SUBREDDIT}")
print(f"  Keyword   : '{KEYWORD}'")
print(f"  Obiettivo : {TARGET_COMMENTS} commenti")

/Users/veronicabosso/PycharmProjects/ELIta_tesi/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Configurazione pronta.
  Subreddit : r/Italia
  Keyword   : 'estate'
  Obiettivo : 700 commenti


## 2. Test connessione API
Attenzione, con servizi intermedi come **Arctic Shift** le risposte possono variare. Se lo status è 200 OK, altrimenti se è 422 riprova dopo qualche secondo.

In [12]:
# Verifica che l'API risponda correttamente prima di procedere
test_url = f"{BASE_URL}/comments/search"
test_params = {
    "subreddit": SUBREDDIT,
    "body"   : KEYWORD,
    "limit"    : 3,
    "after"    : "2022-01-01",
    "before"   : "2022-12-31"
}

resp = requests.get(test_url, headers=HEADERS, params=test_params, timeout=20)
print(f"Status: {resp.status_code}")

if resp.status_code == 200:
    data = resp.json()
    print(f"Risposta OK! Esempio commento:")
    if data.get("data"):
        print(data["data"][0].get("body", "")[:200])
    print(f"\nChiavi disponibili: {list(data['data'][0].keys()) if data.get('data') else 'nessuna'}")
else:
    print(f"Errore: {resp.text}")

Status: 200
Risposta OK! Esempio commento:
Si ma non ci sono più i lunghi e rigidi inverni di una volta; fra un attimo è di nuovo estate

Chiavi disponibili: ['all_awardings', 'archived', 'associated_award', 'author', 'author_created_utc', 'author_flair_background_color', 'author_flair_css_class', 'author_flair_richtext', 'author_flair_template_id', 'author_flair_text', 'author_flair_text_color', 'author_flair_type', 'author_fullname', 'author_patreon_flair', 'author_premium', 'body', 'can_gild', 'collapsed', 'collapsed_because_crowd_control', 'collapsed_reason', 'collapsed_reason_code', 'comment_type', 'controversiality', 'created_utc', 'distinguished', 'edited', 'gilded', 'gildings', 'id', 'is_submitter', 'link_id', 'locked', 'name', 'no_follow', 'parent_id', 'permalink', 'retrieved_on', 'score', 'score_hidden', 'send_replies', 'stickied', 'subreddit', 'subreddit_id', 'subreddit_name_prefixed', 'subreddit_type', 'top_awarded_type', 'total_awards_received', 'treatment_tags', 'unreplia

## 3. Raccolta commenti con paginazione temporale
Questa cella è la più critica, la raccolta di commenti tramite API può essere lenta e soggetta a errori di rete o limiti di rate.

In [13]:
def collect_comments(subreddit, keyword, target=700):
    """
    Raccoglie commenti da Arctic Shift con un limite di 250 per anno.
    """
    all_comments = []
    LIMIT_PER_YEAR = 300  # <-- Limite richiesto

    time_windows = [
        ("2023-01-01", "2024-01-01"),
        ("2022-01-01", "2023-01-01"),
        ("2021-01-01", "2022-01-01"),
        ("2020-01-01", "2021-01-01"),
        ("2019-01-01", "2020-01-01"),
    ]

    for after, before in time_windows:
        # Se abbiamo già raggiunto il target totale generale, possiamo fermarci del tutto
        if len(all_comments) >= target:
            break

        print(f"\nFinestra {after[:4]}: inizio raccolta (max {LIMIT_PER_YEAR})...")

        params = {
            "subreddit": subreddit,
            "body": keyword,
            "limit": 100,
            "after": after,
            "before": before,
            "sort": "desc"
        }

        window_comments = []
        last_utc = None

        while len(window_comments) < LIMIT_PER_YEAR: # <-- Controllo limite annuale
            if last_utc:
                params["before"] = last_utc

            try:
                resp = requests.get(
                    f"{BASE_URL}/comments/search",
                    headers=HEADERS,
                    params=params,
                    timeout=20
                )
                resp.raise_for_status()
                data = resp.json()
            except Exception as e:
                print(f"  Errore: {e}")
                break

            items = data.get("data", [])
            if not items:
                break

            for c in items:
                # Controlliamo il limite anche dentro il loop per non superare i 250
                if len(window_comments) >= LIMIT_PER_YEAR:
                    break

                body = c.get("body", "").strip()
                if body in ("", "[deleted]", "[removed]") or len(body) < 15:
                    continue

                ts = float(c.get("created_utc", 0))
                window_comments.append({
                    "post_id"          : c.get("link_id", "").replace("t3_", ""),
                    "post_title"       : c.get("link_title", ""),
                    "comment_id"       : c.get("id", ""),
                    "comment_text"     : body,
                    "comment_author"   : c.get("author", ""),
                    "comment_timestamp": datetime.utcfromtimestamp(ts).isoformat(),
                    "comment_score"    : c.get("score", 0),
                    "subreddit"        : c.get("subreddit", subreddit),
                    "permalink"        : "https://reddit.com" + c.get("permalink", "")
                                         if c.get("permalink") else ""
                })

            # Prepara il cursore per la pagina successiva
            oldest_utc = min(float(c.get("created_utc", 0)) for c in items)
            last_utc = datetime.utcfromtimestamp(oldest_utc).strftime("%Y-%m-%dT%H:%M:%S")

            print(f"  Scaricati {len(window_comments)} commenti per il {after[:4]}...", end="\r")

            if len(items) < 100:
                break

            time.sleep(0.8)

        # Aggiungiamo i commenti dell'anno alla lista generale
        all_comments.extend(window_comments)
        print(f"  -> Concluso {after[:4]}: {len(window_comments)} commenti. Totale generale: {len(all_comments)}")
        time.sleep(1)

    return all_comments

print("Funzione definita con limite di 300 per anno. Avvio...")
raw_comments = collect_comments(SUBREDDIT, KEYWORD, target=TARGET_COMMENTS)

Funzione definita con limite di 300 per anno. Avvio...

Finestra 2023: inizio raccolta (max 300)...
  -> Concluso 2023: 300 commenti. Totale generale: 300

Finestra 2022: inizio raccolta (max 300)...
  Errore: 422 Client Error: Unprocessable Entity for url: https://arctic-shift.photon-reddit.com/api/comments/search?subreddit=Italia&body=estate&limit=100&after=2022-01-01&before=2022-12-06T12%3A27%3A58&sort=desc
  -> Concluso 2022: 200 commenti. Totale generale: 500

Finestra 2021: inizio raccolta (max 300)...
  -> Concluso 2021: 72 commenti. Totale generale: 572

Finestra 2020: inizio raccolta (max 300)...
  -> Concluso 2020: 50 commenti. Totale generale: 622

Finestra 2019: inizio raccolta (max 300)...
  Errore: 422 Client Error: Unprocessable Entity for url: https://arctic-shift.photon-reddit.com/api/comments/search?subreddit=Italia&body=estate&limit=100&after=2019-01-01&before=2020-01-01&sort=desc
  -> Concluso 2019: 0 commenti. Totale generale: 622


## 4. Arricchimento con titoli post (opzionale)

In [15]:
# Se link_title non è disponibile nei commenti, recuperiamo i titoli dei post
# cercando i post per ID tramite Arctic Shift

df_raw = pd.DataFrame(raw_comments)
missing_titles = df_raw[df_raw["post_title"] == ""]["post_id"].unique()
print(f"Post con titolo mancante: {len(missing_titles)}")

if len(missing_titles) > 0 and len(missing_titles) <= 200:
    # Recuperiamo i titoli a batch di 50
    title_map = {}
    batch_size = 50
    for i in range(0, len(missing_titles), batch_size):
        batch = missing_titles[i:i+batch_size]
        ids_str = ",".join(batch)
        try:
            resp = requests.get(
                f"{BASE_URL}/posts/ids",
                headers=HEADERS,
                params={"ids": ids_str},
                timeout=20
            )
            if resp.status_code == 200:
                for post in resp.json().get("data", []):
                    title_map[post["id"]] = post.get("title", "")
        except Exception as e:
            print(f"  Errore recupero titoli: {e}")
        time.sleep(0.5)
    
    # Aggiorniamo i titoli mancanti
    df_raw["post_title"] = df_raw.apply(
        lambda r: title_map.get(r["post_id"], r["post_title"]) if r["post_title"] == "" else r["post_title"],
        axis=1
    )
    print(f"Titoli recuperati: {len(title_map)}")
else:
    print("Tutti i titoli già presenti (o troppi post da recuperare).")

df_raw

Post con titolo mancante: 409
Tutti i titoli già presenti (o troppi post da recuperare).


,post_id,post_title,comment_id,comment_text,comment_author,comment_timestamp,comment_score,subreddit,permalink
0,18vddfx,,kfqv4u2,Premettendo che la funzione del compito a casa...,Silly-Patience3559,2023-12-31T20:42:51,5,Italia,https://reddit.com/r/Italia/comments/18vddfx/t...
1,18vddfx,,kfqp5fw,Per me i compiti delle vacanze di Natale sono...,ViaNocturna664,2023-12-31T20:06:37,5,Italia,https://reddit.com/r/Italia/comments/18vddfx/t...
2,18ve7pt,,kfqkqtu,Sul nove facevano qualche estate fa all'ora di...,sBrrtou97,2023-12-31T19:39:58,2,Italia,https://reddit.com/r/Italia/comments/18ve7pt/m...
3,18s1f7r,,kfpuon6,"Anche io ho amici in valsassina, tutti over 50...",seth_golden_apple,2023-12-31T17:00:12,1,Italia,https://reddit.com/r/Italia/comments/18s1f7r/a...
4,18v62vn,,kfpqq32,A me è capitato quest'estate di perderlo per q...,barring__,2023-12-31T16:34:40,1,Italia,https://reddit.com/r/Italia/comments/18v62vn/s...
...,...,...,...,...,...,...,...,...,...
617,fcbf9e,,fjg6ljm,"Direi che sia già sulla buona strada, purtropp...",Lucaspo67,2020-03-04T09:25:37,1,Italia,https://reddit.com/r/Italia/comments/fcbf9e/tu...
618,fcbf9e,,fjedno5,Se arriva a questa estate questo corona virus ...,naxil1981,2020-03-03T21:12:30,1,Italia,https://reddit.com/r/Italia/comments/fcbf9e/tu...
619,f8zvhg,,fipxrx1,"Ciao, personalmente ti posso consigliare l'ort...",lostinanotherplanet,2020-02-25T10:35:40,2,Italia,https://reddit.com/r/Italia/comments/f8zvhg/we...
620,ez6o7k,,fglqjlz,“Il meme ok boomer è diventato virale tra gli ...,metronomo167,2020-02-05T13:10:43,2,Italia,https://reddit.com/r/Italia/comments/ez6o7k/no...


## 5. Salvataggio CSV

In [16]:
df_corpus = df_raw.drop_duplicates(subset="comment_id").reset_index(drop=True)
df_corpus.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"Salvati {len(df_corpus)} commenti unici in '{OUTPUT_CSV}'")
print(f"Colonne: {list(df_corpus.columns)}")
df_corpus[["post_title", "comment_text", "comment_author", "comment_timestamp"]].head(5)

Salvati 622 commenti unici in 'corpus_Italia_estate.csv'
Colonne: ['post_id', 'post_title', 'comment_id', 'comment_text', 'comment_author', 'comment_timestamp', 'comment_score', 'subreddit', 'permalink']


,post_title,comment_text,comment_author,comment_timestamp
0,,Premettendo che la funzione del compito a casa...,Silly-Patience3559,2023-12-31T20:42:51
1,,Per me i compiti delle vacanze di Natale sono...,ViaNocturna664,2023-12-31T20:06:37
2,,Sul nove facevano qualche estate fa all'ora di...,sBrrtou97,2023-12-31T19:39:58
3,,"Anche io ho amici in valsassina, tutti over 50...",seth_golden_apple,2023-12-31T17:00:12
4,,A me è capitato quest'estate di perderlo per q...,barring__,2023-12-31T16:34:40


## 6. NLP: Tokenizzazione, Lemmatizzazione e POS-tagging
Modello italiano `it_core_news_sm` di spaCy.

In [17]:
import spacy

nlp = spacy.load("it_core_news_sm")

def process_text(text):
    doc = nlp(str(text))
    return [
        {
            "token" : token.text,
            "lemma" : token.lemma_.lower(),
            "pos"   : token.pos_,
            "is_adj": token.pos_ == "ADJ"
        }
        for token in doc
        if not token.is_space and not token.is_punct
    ]

# Test su un commento di esempio
esempio = df_corpus["comment_text"].iloc[0]
print(f"Commento:\n{esempio}\n")
print("Analisi:")
for t in process_text(esempio):
    flag = " <- ADJ" if t["is_adj"] else ""
    print(f"  {t['token']:20s} | {t['lemma']:20s} | {t['pos']}{flag}")

Commento:
Premettendo che la funzione del compito a casa sia fissare le conoscenze apprese in classe, ammetto che molti colleghi le usano per sopperire alla mancata applicazione in classe (vuoi perchè il tempo è poco, vuoi perché i tempi di apprendimento di 25 e passa studenti sono difficili da armonizzare e quindi si sta 40 minuti su un singolo argomento). Mai assegnato compiti per le vacanze, li trovo una presa in giro: nel migliore dei casi, verranno fatti in fretta e furia 1/2 giorni prima della vacanze, nel peggiore da mamma e papà. Un altro discorso è l'estate dove un pochino di compiti ci sta bene.

Analisi:
  Premettendo          | premettere           | VERB
  che                  | che                  | SCONJ
  la                   | il                   | DET
  funzione             | funzione             | NOUN
  del                  | di il                | ADP
  compito              | compito              | NOUN
  a                    | a                    | ADP
  casa  

In [18]:
from tqdm.auto import tqdm

print(f"Processamento di {len(df_corpus)} commenti con spaCy...")

all_tokens = []
for _, row in tqdm(df_corpus.iterrows(), total=len(df_corpus)):
    for t in process_text(row["comment_text"]):
        all_tokens.append({"comment_id": row["comment_id"], **t})

df_tokens = pd.DataFrame(all_tokens)
df_adj    = df_tokens[df_tokens["is_adj"] == True]

print(f"\nToken totali     : {len(df_tokens)}")
print(f"Aggettivi trovati: {len(df_adj)}")
print(f"\nTop 20 aggettivi più frequenti:")
print(df_adj["lemma"].value_counts().head(20))

/Users/veronicabosso/PycharmProjects/ELIta_tesi/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processamento di 622 commenti con spaCy...


100%|██████████| 622/622 [00:12<00:00, 50.36it/s]


Token totali     : 66867
Aggettivi trovati: 4250

Top 20 aggettivi più frequenti:
lemma
altro        77
primo        71
stesso       68
bello        54
nuovo        50
pieno        47
grande       40
buono        39
possibile    38
diverso      35
libero       34
alto         34
vero         34
scorso       32
caldo        31
ultimo       31
elettrico    31
giusto       30
maggiore     29
migliore     27
Name: count, dtype: int64


In [19]:
tokens_file = f"tokens_{SUBREDDIT}_{KEYWORD}.csv"
df_tokens.to_csv(tokens_file, index=False, encoding="utf-8-sig")
print(f"Token salvati in '{tokens_file}'")

Token salvati in 'tokens_Italia_estate.csv'


## 7. Statistiche del corpus

In [20]:
import plotly.express as px

print("=" * 50)
print("STATISTICHE CORPUS")
print("=" * 50)
print(f"Commenti totali          : {len(df_corpus)}")
print(f"Post unici               : {df_corpus['post_id'].nunique()}")
print(f"Autori unici             : {df_corpus['comment_author'].nunique()}")
print(f"Token totali             : {len(df_tokens)}")
print(f"Aggettivi totali         : {len(df_adj)}")
print(f"Aggettivi unici (lemmi)  : {df_adj['lemma'].nunique()}")
lunghezze = df_corpus["comment_text"].str.split().str.len()
print(f"Lunghezza media commenti : {lunghezze.mean():.1f} parole")
print(f"Periodo temporale        : {df_corpus['comment_timestamp'].min()[:10]} → {df_corpus['comment_timestamp'].max()[:10]}")

# Distribuzione per anno
df_corpus["anno"] = df_corpus["comment_timestamp"].str[:4]
fig1 = px.bar(
    df_corpus["anno"].value_counts().sort_index().reset_index(),
    x="anno", y="count",
    title="Commenti per anno",
    labels={"anno": "Anno", "count": "Numero commenti"}
)
fig1.show()

# Distribuzione lunghezza commenti
fig2 = px.histogram(
    x=lunghezze, nbins=40,
    title="Distribuzione lunghezza commenti (parole)",
    labels={"x": "Parole", "y": "Commenti"}
)
fig2.show()

STATISTICHE CORPUS
Commenti totali          : 622
Post unici               : 409
Autori unici             : 504
Token totali             : 66867
Aggettivi totali         : 4250
Aggettivi unici (lemmi)  : 1551
Lunghezza media commenti : 104.9 parole
Periodo temporale        : 2020-01-16 → 2023-12-31
